# Learn representations with I-JEPA on Oxford Flowers

You will train an I-JEPA encoder on Oxford Flowers 102, a ViT that sees the visible patches of an image and predicts, in representation space, the embeddings of the patches that were hidden. No pixel reconstruction, no augmentation pipeline. After training you will measure the representations with a linear probe and a kNN probe, watch the collapse telemetry over the run, and pull the nearest neighbours of query images out of the frozen embeddings.

I-JEPA trains by prediction in latent space. A context encoder sees only some patches, a target encoder sees the whole image, and a narrow predictor maps the context embeddings plus the target positions to the target embeddings. The target encoder is an exponential moving average of the context encoder rather than a trained module, so the targets keep improving as the encoder does and there is no pixel-level shortcut to lower the loss.

**Expected time.** About 25 minutes on an A100, 30 to 40 on a TPU v5e, and 45 to 60 on an RTX 4080, for 100 epochs of 124 steps at 224 px with a ViT-S width encoder. A CPU run is not realistic at this configuration.

In [ ]:
# Install cell: only runs on Colab (import google.colab succeeds there).
# A TPU runtime gets jax[tpu] instead of jax[cuda12]. Locally this cell does nothing,
# and the figures below need matplotlib and tensorflow-datasets in your environment.
import os
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ModuleNotFoundError:
    IN_COLAB = False

if IN_COLAB:
    jax_spec = "jax[tpu]" if "COLAB_TPU_ADDR" in os.environ else "jax[cuda12]"
    %pip install -q "dew-ml[tfds] @ git+https://github.com/AshishKumar4/dew" {jax_spec} tensorflow-datasets matplotlib

In [ ]:
# Every size knob in one place.
IMAGE_SIZE = 224        # pixels; patch 16 gives the 14x14 grid of the paper
PATCH_SIZE = 16
BATCH_SIZE = 64
EPOCHS = 100
LEARNING_RATE = 1e-3
EMB_FEATURES = 384      # encoder width (ViT-S)
NUM_LAYERS = 12
NUM_HEADS = 6
NUM_TARGET_BLOCKS = 4  # blocks the encoder must predict, hidden from it
PROBE_CLASSES = 102
VAL_RECORDS = 512       # held out from the head of the dataset, canonical order
RETRIEVAL_IMAGES = 256  # held-out images embedded for the nearest-neighbour grid
OUT_DIR = "flowers-jepa"

In [ ]:
import jax
print("devices:", jax.devices())
print("backend:", jax.default_backend())

## The data

Oxford Flowers 102 is 8189 labelled flower photographs, downloaded from TensorFlow Datasets on first use. The grain loader resizes each record to 224 px and carries its class index through as `label`, which is what the probes will score against. The first `VAL_RECORDS` images in canonical order are the validation split, so the probes never score an image the encoder trained on.

In I-JEPA the variation comes from the masking, so this loader applies no flips and no colour jitter.

In [ ]:
import cv2
import numpy as np
from dew.data.dataloaders import get_dataset_grain
from dew.data.sources.images import ImageTFDSSource

data = get_dataset_grain("oxford_flowers102", batch_size=BATCH_SIZE,
                         image_scale=IMAGE_SIZE, worker_count=2,
                         val_count=VAL_RECORDS, val_batch_size=256)
print("train:", data["train_len"], "val:", data["val_len"])

batch = next(iter(data["train"]()))
print(batch["image"].shape, batch["image"].dtype, "labels:", batch["label"][:8])

# The same TFDS source the loader reads through, indexed directly: records 0
# to RETRIEVAL_IMAGES are the head of the dataset, which is exactly the slice
# get_dataset_grain holds out as validation. Reading them here needs no second
# loader and no worker processes, and these are the images the retrieval grid
# at the end of the notebook embeds.
source = ImageTFDSSource(name="oxford_flowers102", use_tf=False).get_source(None)
records = [source[i] for i in range(RETRIEVAL_IMAGES)]
val_images = np.stack([
    cv2.resize(np.asarray(r["image"]), (IMAGE_SIZE, IMAGE_SIZE),
               interpolation=cv2.INTER_AREA) for r in records])
val_labels = np.asarray([int(r["label"]) for r in records])
print("retrieval images:", val_images.shape)

## The mask

Each image gets `NUM_TARGET_BLOCKS` rectangular blocks of patches to predict, drawn with a random scale and aspect ratio in the ranges the I-JEPA paper uses, and the context is a random subset of everything left. The geometry is resolved once for the 14x14 grid and each step samples block shapes and positions from it, so every mask has the same shape and the training step stays compiled.

The figure below is one sampled mask. The encoder sees the context patches and predicts the embeddings of the target blocks, and it is told where those blocks are without being shown what is in them.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from dew.objectives.jepa import multi_block_mask

GRID = (IMAGE_SIZE // PATCH_SIZE, IMAGE_SIZE // PATCH_SIZE)
mask = multi_block_mask(GRID, num_targets=NUM_TARGET_BLOCKS)
print("grid", GRID, "| context tokens:", mask.num_context,
      "| targets:", mask.num_targets, "blocks x", mask.block_area, "tokens each")

context_idx, target_idx = mask.sample(jax.random.PRNGKey(0), 1)
grid_view = np.zeros(GRID[0] * GRID[1])
grid_view[context_idx[0]] = 1
grid_view[target_idx.reshape(-1)] = 2
plt.figure(figsize=(3, 3))
plt.imshow(grid_view.reshape(GRID), cmap="coolwarm", interpolation="nearest")
plt.title("blue: context   red: targets")
plt.axis("off")
plt.show()

## Encoder, predictor, target encoder

The encoder is a ViT (`jepa_encoder`). The predictor is a narrower transformer (`jepa_predictor`) that reads the context embeddings plus mask tokens standing in for the targets, and outputs embeddings at the encoder's width. The target encoder has no parameters of its own. It is the EMA copy of the context encoder that the trainer already maintains, and the objective's `EMASpec` restricts that average to the `context_encoder` subtree so the predictor stays out of it.

The loss is the mean squared distance between predictions and layer-normalized targets, in fp32. The layer norm has no learned affine, which fixes the scale of the prediction problem, so shrinking the embeddings does not lower the loss. Two telemetry numbers ship with every step, because this objective fails silently. `repr_std` is how much the embeddings vary across a batch; it goes to zero exactly when the encoder stops distinguishing inputs. `repr_cov_offdiag` is the magnitude of the off-diagonal covariance; it rises when dimensions become redundant, which can happen while `repr_std` still looks healthy.

In [ ]:
import optax
from dew.inputs import DiffusionInputConfig
from dew.objectives.jepa import JepaObjective
from dew.objectives.jepa.probes import get_knn_probe_metric, get_linear_probe_metric
from dew.registry import apply_precision_policy, build_model

encoder_config = apply_precision_policy("jepa_encoder", dict(
    patch_size=PATCH_SIZE, emb_features=EMB_FEATURES,
    num_layers=NUM_LAYERS, num_heads=NUM_HEADS,
), dtype="bfloat16", attention_impl="auto")
encoder = build_model("jepa_encoder", encoder_config)
predictor = build_model("jepa_predictor", dict(
    grid=GRID, emb_features=EMB_FEATURES, predictor_features=EMB_FEATURES // 2,
    num_layers=NUM_LAYERS // 2, num_heads=NUM_HEADS,
    dtype=encoder_config["dtype"], attention_impl=encoder_config["attention_impl"]))

objective = JepaObjective(encoder, predictor, mask=mask,
                          sample_data_key="image",
                          sample_data_shape=(IMAGE_SIZE, IMAGE_SIZE, 3))

## Training, with the telemetry recorded

The trainer is the one the diffusion and language model notebooks use. What it runs at the end of every epoch is the objective's validation step, which embeds a batch of held-out images with the frozen EMA encoder, and then every `EvaluationMetric` the run was given.

The collapse telemetry goes in through that seam. Each of the two metrics below computes one health number on those embeddings and appends it to a list, so the run can be plotted afterwards. `higher_is_better` only decides which value the trainer keeps as the best one; both numbers are recorded here so they can be plotted.

The probes are not in this list, and the reason is worth knowing. A validation batch from this loader repeats records (256 rows carry 44 distinct labels, the most common of them twelve times), so a probe fit on half of one batch is scored on copies of the images it was fit on, and it reads far higher than the representation deserves. The probes run once after training instead, on distinct images.

In [ ]:
from dew.eval.common import EvaluationMetric
from dew.objectives.jepa import representation_health
from dew.training import ObjectiveTrainer

std_history, cov_history = [], []

def record_std(embeddings, batch):
    value = float(representation_health(embeddings)["repr_std"])
    std_history.append(value)
    return value

def record_cov(embeddings, batch):
    value = float(representation_health(embeddings)["repr_cov_offdiag"])
    cov_history.append(value)
    return value

trainer = ObjectiveTrainer(
    encoder, optax.adamw(LEARNING_RATE), objective=objective,
    input_config=DiffusionInputConfig(sample_data_key="image",
                                       sample_data_shape=(IMAGE_SIZE, IMAGE_SIZE, 3),
                                       conditions=[]),
    eval_metrics=[
        EvaluationMetric(record_std, name="repr_std", higher_is_better=True),
        EvaluationMetric(record_cov, name="repr_cov_offdiag", higher_is_better=False),
    ],
    rngs=jax.random.PRNGKey(0), name=OUT_DIR,
    checkpoint_base_path="./checkpoints/flowers-jepa")
state = trainer.fit(data, training_steps_per_epoch=data["train_len"] // BATCH_SIZE,
                    epochs=EPOCHS, val_steps_per_epoch=2)

## What the curves say

`repr_std` is the per-dimension spread of the embeddings across a batch, and it reaches zero exactly when the encoder has collapsed to a constant, so it should stay well away from zero. `repr_cov_offdiag` is the magnitude of the off-diagonal covariance; it rises when the embedding uses fewer directions than it has, which is the other way this objective fails. Pass 0 in both plots is the sanity validation the trainer runs before training, so it is the untrained baseline.

In [ ]:
def per_pass(history):
    """One point per validation pass, averaged over that pass's batches."""
    values = np.asarray(history)
    return values.reshape(-1, values.size // (EPOCHS + 1)).mean(axis=1)

std_curve, cov_curve = per_pass(std_history), per_pass(cov_history)
passes = np.arange(len(std_curve))

fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].plot(passes, std_curve)
axes[0].axhline(std_curve[0], linestyle="--", color="gray")
axes[0].set_title("repr_std (away from 0 is healthy)")
axes[1].plot(passes, cov_curve)
axes[1].axhline(cov_curve[0], linestyle="--", color="gray")
axes[1].set_title("repr_cov_offdiag (low is healthy)")
for ax in axes:
    ax.set_xlabel("validation pass (0 is before training)")
plt.tight_layout()
plt.savefig("jepa-telemetry.png", dpi=110)
plt.show()
print("wrote jepa-telemetry.png |",
      f"repr_std {std_curve[0]:.3f} -> {std_curve[-1]:.3f} |",
      f"repr_cov_offdiag {cov_curve[0]:.4f} -> {cov_curve[-1]:.4f}")

## Probing the representation

A probe asks how much of a label a frozen encoder already knows. `get_linear_probe_metric` and `get_knn_probe_metric` are the two the JEPA recipe uses, and both take the pooled embeddings and the labels of a set, fit a classifier on the first half and score it on the second. Chance on 102 classes is just under 1 percent.

The shuffled-label control beside each number is what makes them readable. A probe with more embedding dimensions than samples can fit anything it is shown, so the honest question is how far the real-label score sits above the same probe on permuted labels.

In [ ]:
from dew.objectives.jepa.probes import get_knn_probe_metric, get_linear_probe_metric

embed = objective.make_validation_step()
probe_embeddings = np.asarray(embed(state, {"image": val_images}))
shuffled = np.random.RandomState(0).permutation(val_labels)

for metric in (get_linear_probe_metric(PROBE_CLASSES), get_knn_probe_metric(PROBE_CLASSES)):
    real = float(metric.function(probe_embeddings, {"label": val_labels}))
    control = float(metric.function(probe_embeddings, {"label": shuffled}))
    print(f"{metric.name}: {real:.3f} on the real labels, {control:.3f} on shuffled labels")

## Nearest neighbours

The point of the encoder is the embedding space, so look at it directly. Pool the frozen EMA encoder's tokens for each validation image, normalise the embeddings, and take cosine neighbours of a few queries. Each row is one query followed by its four nearest neighbours. Flowers of the same species clustering together is the same thing the probes measured numerically.

In [ ]:
# The validation step is the objective's: pooled embeddings from the frozen
# EMA encoder, the same ones the probes scored.
embed = objective.make_validation_step()
embeddings = np.asarray(embed(state, {"image": val_images}))
embeddings = embeddings / (np.linalg.norm(embeddings, axis=-1, keepdims=True) + 1e-8)
print(val_images.shape, embeddings.shape)

queries = [0, 1, 2, 3]
fig, axes = plt.subplots(len(queries), 5, figsize=(12, 2.6 * len(queries)))
for row, q in enumerate(queries):
    similarity = embeddings @ embeddings[q]
    neighbours = np.argsort(-similarity)[1:5]
    for col, idx in enumerate([q, *neighbours]):
        ax = axes[row, col]
        ax.imshow(val_images[idx].astype(np.uint8))
        tag = "query" if col == 0 else f"nn {col}"
        ax.set_title(f"{tag} | label {val_labels[idx]}", fontsize=8)
        ax.axis("off")
plt.tight_layout()
plt.savefig("jepa-retrieval.png", dpi=110)
plt.show()
print("wrote jepa-retrieval.png")

## Keeping the encoder

The thing to keep from a JEPA run is the EMA of the context encoder, without the predictor. `save_params` writes it as a safetensors file under the names the tree uses, so anything that reads safetensors reads it back. The trainer's checkpoints under `./checkpoints/flowers-jepa` hold the full train state, including the optimizer, if you want to resume.

In [ ]:
from pathlib import Path

from dew.interop import save_params

# safetensors writes through a temporary file in the target directory and does
# not create it.
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
save_params(state.ema_params["params"]["context_encoder"], f"{OUT_DIR}/encoder.safetensors")
print("wrote", f"{OUT_DIR}/encoder.safetensors")

## Where to go next

The paper's ViT-H/16 trains 300 epochs on ImageNet with far more target blocks; the knobs are the ones this notebook set once (`NUM_TARGET_BLOCKS`, `EPOCHS`, the model width, the mask scale and aspect ranges in `multi_block_mask`). `recipes/jepa/train.py` runs the same configuration from the command line, and `jepa_video_encoder` with a `factorized=True` predictor does the same job on video clips.